By using the create documnet chain and retrieval chain functions

In [18]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import ChatHuggingFace,HuggingFaceEmbeddings,HuggingFaceEndpoint
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.document_loaders import WebBaseLoader
from dotenv import load_dotenv
from langchain_classic.chains import create_retrieval_chain


In [3]:
load_dotenv()

True

In [4]:
loader=WebBaseLoader("https://docs.smith.langchain.com/tutorials/Administarators/manage_spend")

In [5]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administarators/manage_spend', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content="LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewEngineTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics.LangSmith w

In [6]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=text_splitter.split_documents(docs)

In [7]:
chunks

[Document(metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administarators/manage_spend', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content="LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewEngineTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics.LangSmith w

In [8]:
embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4368.62it/s]


In [9]:
from langchain_community.vectorstores import FAISS
vectorstore=FAISS.from_documents(chunks,embeddings)

In [10]:
query='Add tracing to your app in minutes'
result=vectorstore.similarity_search(query)
result[0].page_content

'Copy the key and save it securely.Once your account and API key are ready, set up tracing:Set up tracingAdd tracing to your app in minutes with environment variables, framework integrations, or the SDK.Trace a RAG applicationFollow a step-by-step tutorial to instrument a retrieval-augmented generation app from start to finish.Investigate and monitorView tracesFilter, export, share, and compare traces via the UI or API.Monitor performanceBuild dashboards and set alerts to track quality and catch issues early.Configure automationsAutomate workflows with rules, webhooks, and online evaluations.Collect feedbackAnnotate outputs and gather user feedback using queues or inline annotation.Find and fix failures with EngineAutomatically detect recurring issues in your traces, diagnose their root cause, and resolve them with LangSmith Engine.For terminology and core concepts, refer to Observability concepts. For trace pricing, retention, and limits, see Usage and billing.To set up a LangSmith'

In [37]:
llm=HuggingFaceEndpoint(
    repo_id='meta-llama/Llama-3.1-8B-Instruct',
    task='text generation'
)
model=ChatHuggingFace(llm=llm)

In [47]:
prompt=ChatPromptTemplate.from_template( """
Answer the following question based on the provided context:\n{context}
Question:{input}                                      
if the context provided is not sufficient to answer the question then simply say I don't know
"""
)


In [ ]:
document_chain=create_stuff_documents_chain(model,prompt)
# document_chain

In [41]:
#you see the above output that is the docs are made into context by format_docs()
# next prompt is genearted 
# the prompt and question and merged and passed to llm
# from llm the answer is passed to parser

In [49]:
# Retriever
retriever=vectorstore.as_retriever()
final_chain=create_retrieval_chain(retriever,document_chain)

In [ ]:
# final_chain

In [51]:
#Get the response from llm
response=final_chain.invoke({'input':'Add tracing to your app in minutes'})
response['answer']

'To set up tracing in your app, you can follow these steps:\n\n1. Create an account and API key on the LangSmith Observability website.\n2. Set up tracing by adding environment variables, framework integrations, or using the SDK.\n3. Follow the step-by-step tutorial to instrument a retrieval-augmented generation app from start to finish.\n\nThis should allow you to add tracing to your app in minutes.'